# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print("Dataset name:", metadata.get('name', 'N/A'))
print("Description:", metadata.get('description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll display all available record set `@id`s, with their field and column `@id`s, as per the schema.

In [ ]:
# List available record sets, their @id, and field @ids
from pprint import pprint

print("Available Record Sets in the dataset:")
if hasattr(dataset, "record_sets"):
    record_sets = dataset.record_sets
else:
    # fallback: try reading from the metadata 
    record_sets = metadata.get('recordSet', [])

record_set_ids = []

for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs['@id']}   Name: {rs.get('name','N/A')}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields and columns:")
    for field in fields:
        field_id = field.get('@id') if isinstance(field, dict) else field
        print(f"    - Field @id: {field_id}")
        # Show columns for this field, if available
        if isinstance(field, dict):
            columns = field.get('column', [])
            if not isinstance(columns, list):
                columns = [columns]
            for col in columns:
                print(f"      - Column @id: {col}")

if not record_set_ids:
    print("No record sets found. Please check the schema or mlcroissant version.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by @id
import warnings

record_sets = record_set_ids  # Collected above
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head(2))
    except Exception as e:
        warnings.warn(f"Could not load data for {record_set_id}: {str(e)}")

# For demonstration, pick the first record set as main (if any exists)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Using main record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data. We'll select a numeric field and a grouping field from our main record set. Please refer to the list of columns printed in the previous cell to update the variables if needed.

In [ ]:
import numpy as np

# Show available columns
if main_record_set_id:
    print("Columns in main DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())

    # Example: Try to select a likely numeric field
    # (Columns are referenced by their @id as per Croissant)
    numeric_candidate_keys = [
        col for col in dataframes[main_record_set_id].columns
        if any(key in col.lower() for key in ['abundance', 'count', 'coverage', 'intensity', 'amount', 'mass', 'weight'])
    ]
    print("Numeric field candidates:", numeric_candidate_keys)
    if numeric_candidate_keys:
        numeric_field = numeric_candidate_keys[0]
    else:
        # Default: Take the first column
        numeric_field = dataframes[main_record_set_id].columns[0]

    print(f"Selected numeric field for EDA: {numeric_field}")

    df = dataframes[main_record_set_id]
    # Force numeric, errors='coerce' will turn invalid parsing into NaN
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean value):")
    display(filtered_df.head())

    # Normalize
    mu = filtered_df[numeric_field].mean()
    sigma = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma if sigma > 0 else 0
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Guess a group field (a string, category, etc)
    group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
    print("Group-by field candidates:", group_candidates)
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If grouping field exists and is reasonable
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded Croissant schema and dataset metadata using `mlcroissant`.
- Discovered available record sets and their fields using their `@id`s.
- Loaded records into DataFrames and performed example filtering and normalization on a representative numeric field.
- Explored basic grouping and plotted distributions to help understand the structure and value ranges in the dataset.

This notebook serves as a template for further, deeper exploration of datasets described by Croissant schemas using the `mlcroissant` tool. For custom or more complex analyses, consult the Croissant schema documentation for field meanings and custom processing strategies.